### SHM data generation in parallel
<small>CONTENT: Example for D1 - 
EVs + OVs - 3 years/36 months (2020-2022) - 26280 samples expected </small>

#### Setup

In [ ]:
# IMPORT
from math import sqrt
import os
from statistics import *
import numpy as np
import pandas as pd
import h5py
import scipy.stats as stats
from scipy.signal import butter, filtfilt, welch
from scipy.integrate import odeint
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import ray

In [ ]:
# FOLDER
mainFolder = os.getcwd()
mainFolder = mainFolder.replace("\\", "/")
inputFolder = mainFolder + "/input"
outputFolder = mainFolder + "/output"

if not os.path.exists(inputFolder):
	os.makedirs(inputFolder)
if not os.path.exists(outputFolder):
	os.makedirs(outputFolder)

#### Simulated monitoring - 3 years - in parallel using Ray
Single example for undamaged condition

In [ ]:
# LOAD KM
datasetFolder = outputFolder + "/datasets"
loadFileName = "/1_km_UD.txt"
loadFilePath = datasetFolder + loadFileName
with open(loadFilePath,'r') as f:
	KM = pd.read_csv(f, delimiter = "\t", header = None)
	colNames = ["k", "m"]
KM.columns = colNames
KM = KM.astype('float64')

In [ ]:
# OTHER INPUT
g = 9.81 # m/s²
zeta = 0.05  # Damping ratio
cA = [0,0,0] # [x,y,theta]
sigma_eps = 0.000001

In [ ]:
# TIMESERIES 3 YEARS
start_date = datetime(2020,1,1,0,0,0) 		# Start date of recording: 01/01/2020 00:00
end_date = datetime(2023,12,31,23,0,0) 		# End date before just to try
intVal = 1 									# Interval of acquisition
offsetAlias = 'H'							# Unit of measument of acquisition interval [hours]
tHourly = pd.date_range(start = start_date, end = end_date, freq = ('{}{}'.format(intVal, offsetAlias)))
# erase feb 29 in 2020 and 2024
timeSimulation = tHourly[~((tHourly.month == 2) & (tHourly.day == 29))]
daysMonitored = len(pd.Series(timeSimulation.date).unique())
timeSimulationStr = timeSimulation.strftime('%Y-%m-%d %H:%M:%S').to_numpy()

In [ ]:
# SIMULATION INPUT
# units kg,m,s
days = 365
hours = 24
years = 4
hours1year = days * hours
daysTot = days * years
timeSteps = days * hours * years
timeSeries = np.linspace(0, timeSteps-1, timeSteps)
time = np.arange(0, daysTot, 1/hours)
nSim = 5 # maximum number of simulations for data generation
timeStepsMax = days * hours * years * nSim
timeSeriesMax = np.linspace(0, (timeStepsMax)-1, (timeStepsMax))

# single AV
fs = 100 # Hz
minutes = 3
acquisLength = minutes * 60  # acquisition length in seconds (5mins)
nSamples = (fs * acquisLength)  # total number of samples
dt = 1/fs
t = np.linspace(0, acquisLength, nSamples)  # time vector

In [ ]:
# STATE-SPACE SOLUTION
# Methods to run state space solution based on m, k, bc and input parameters + noise
# function to solve the state space equantions
def stateSpaceEquations(x, t, A, B, G, u, wgn_eps, fs):
	idx = min(int(t * fs), len(u) - 1)  # make sure to take the right time index
	dxdt = np.dot(A, x) + B.flatten() * u[idx] + G.flatten() * wgn_eps[idx]
	return dxdt
# function to construct the state space equations and run the simulation
def stateSpaceSolution (k, m, zeta, bc, acquisLength, nSamples, inputAV, sigma_eps):
	omega_n = np.sqrt(k / m)  # Natural frequency in rad/s
	c = 2 * zeta * m * omega_n # N s/m 
	fs = nSamples/acquisLength
	dt = 1/fs
	A = np.array([[0, 1], [-k/m, -c/m]]) # System matrix
	B = np.array([[0], [1/m]]) # input matrix for deterministic inputs
	G = np.array([[0], [1/m]]) # input matrix for stochastic inputs (noise)
	#C = np.array([[1, 0], [0, 1]]) # output matrix, assuming full-state observation
	#D = np.array([[0], [0]]) # direct transmission matrix, often zero in mechanical systems
	# INITIAL CONDITIONS
	x0 = np.array([bc[0], bc[1]])  # initial displacement and velocity
	# TIME VECTOR
	t = np.linspace(0, acquisLength, nSamples)  # using your existing time definitions
	# INPUT (FORCING FUNCTION)
	u = inputAV
	# INPUT UNCERTAINTY
	mu_eps = 0  # Mean of the noise
	wgn_eps = np.random.normal(mu_eps, sigma_eps, (nSamples, 1))
	# SOLVING THE STATE-SPACE EQUATIONS
	sol = odeint(stateSpaceEquations, x0, t, args=(A, B, G, u, wgn_eps, fs))
	displacement = sol[:, 0]
	velocity = sol[:, 1]
	acceleration = np.diff(velocity) / dt
	acceleration = np.append(acceleration, acceleration[-1])  # to maintain the array size
	return t, displacement, velocity, acceleration

In [ ]:
# GLOBAL VARIABLES BEFORE RAY
mu = 0  # Mean
factor = 3
sigma = 0.0001 / factor # Standard deviation in m/s^2
fs = 100 # Hz
minutes = 3
acquisLength = minutes * 60  # acquisition length in seconds (5mins)
nSamples = (fs * acquisLength)  # total number of samples
order = 3  # Filter order
cases = 2
frequencies = {'human': (1.2, 4.8), 'traffic': (7, 15)}  #'wind': (0.1, 0.3)
amplitudes = {'human': (5e-6, 5e-4), 'traffic': (1e-6, 2e-4)} #'wind': (0.0002, 0.002)
factor = 3
g = 9.81 # m/s²
zeta = 0.05  # Damping ratio
cA = [0,0,0] # [x,y,theta]
sigma_eps = 0.000001
nSimMax = 10 # maximum number of simulations for data generation
fTol = 0.1 #Hz tolerance
nperseg = 2**12
noverlap = nperseg * 0.5

In [ ]:
# GLOBAL METHODS BEFORE RAY
def generate_input(nSamples, amplitudes, frequencies, factor, order, fs, mu, g, m):
	randHumanSigma = np.random.uniform(low=amplitudes['human'][0]/factor, high=amplitudes['human'][1]/factor)
	wgnHuman = np.random.normal(mu, randHumanSigma, nSamples)
	filCoeffHuman = butter(order, [frequencies['human'][0]/(0.5*fs), frequencies['human'][1]/(0.5*fs)], btype='bandpass')
	bandPassHuman = filtfilt(filCoeffHuman[0], filCoeffHuman[1], wgnHuman)
	randTrafficSigma = np.random.uniform(low=amplitudes['traffic'][0]/factor, high=amplitudes['traffic'][1]/factor)
	wgnTraffic = np.random.normal(mu, randTrafficSigma, nSamples)
	filCoeffTraffic = butter(order, [frequencies['traffic'][0]/(0.5*fs), frequencies['traffic'][1]/(0.5*fs)], btype='bandpass')
	bandPassTraffic = filtfilt(filCoeffTraffic[0], filCoeffTraffic[1], wgnTraffic)
	return (bandPassHuman + bandPassTraffic) * m * g

def extractFreq(acc,fs,nperseg,noverlap):
	freqs, psd = welch(acc, fs, window = 'hann', nperseg = nperseg, noverlap = noverlap)
	peakInd = np.argmax(psd)
	extractedFreq = freqs[peakInd]
	return extractedFreq

def checkFreq(freq, freqRef, Tol):
	if abs(freqRef-freq) <= Tol:
		return True
	else:
		return False

In [ ]:
# GENERATE DATA FOR 1st DATASET
outputFolderD1 = outputFolder + "/acceleration" + "/D1"

In [ ]:
# RUN DYNAMIC DATA GENERATION IN RAY
num_cpus = os.cpu_count()
ray.init(num_cpus = num_cpus-2)

@ray.remote
def runSSsimulationAcc(i, k, m, zeta, cA, acquisLength, nSamples, sigma_eps, outputFolder, nSimMax):
	# Reference analytical frequency for current simulation
	freqRef = (np.sqrt(k / m)) / (2 * np.pi)
	# Init lists and counter
	accSelect = [] # list to save acc data
	freq = [] # list to save frequency data
	sim = 0
	while sim < nSimMax:
		# Generate input for current simulation
		try:
			inputTemp = generate_input(nSamples, amplitudes, frequencies, factor, order, fs, mu, g, m)
		except Exception as e:
			print(f"Error in input generation with index {i}: {e}")
			raise e
		# Utilize the provided stateSpaceSolution function
		try:
			_, _, _, a = stateSpaceSolution(k, m, zeta, cA, acquisLength, nSamples, inputTemp, sigma_eps)
		except Exception as e:
			print(f"Error in simulation with index {i}: {e}")
			raise e
		# Extract frequency and check tolerace, if not satisfied continue
		try:
			extractedFreq = extractFreq(a,fs,nperseg,noverlap)
			freq.append(extractedFreq)
			freqInd = freq.index(extractedFreq)
			accSelect.append(a)
			if checkFreq(extractedFreq, freqRef, fTol) == True:
				accSelect = [accSelect[freqInd]]
				break
			sim += 1
		except Exception as e:
			print(f"Error in frequency extraction with index {i}: {e}")
			raise e
	if sim == nSimMax:
		bestFreq = min(freq, key = lambda x: abs(x - freqRef))
		bestInd = freq.index(bestFreq)
		accSelect = [accSelect[bestInd]]
	# Prepare the data to be saved to file
	try:
		print(f"Saving file {i}")
		iter_str = str(i).zfill(5)
		fileName = os.path.join(outputFolder, f"acc{iter_str}-1.h5").replace("\\", "/")
		accfloat32 = np.array(accSelect, dtype = np.float32)
		with h5py.File(fileName, 'w') as f:
			f.create_dataset('acc', data = accfloat32)
	except Exception as e:
		print(f"Error in file writing with index {i}: {e}")
		raise e
	return

futures = [runSSsimulationAcc.remote(i, KM.iloc[i, 0], KM.iloc[i, 1],
									 zeta, cA, acquisLength, nSamples, sigma_eps, 
									 outputFolderD1, nSimMax) for i in range(len(timeSimulation))]

results = ray.get(futures) 
ray.shutdown() # shutdown ray
